# MCRAH — Colab Training Notebook

Motion-Coherent Rigidity-Adaptive Hypergraph for Dynamic 3D Gaussian Splatting (CVPR 2027).

This notebook runs the full MCRAH pipeline on a Colab GPU runtime:
1. Clone the repo from GitHub
2. Install Python deps + Rust toolchain (for the fast data preprocessor)
3. Download the D-NeRF dataset
4. Train on one category (defaults to `trex`)
5. Evaluate NVS metrics + rollout stability

**Runtime requirement:** GPU (T4 or better). Select
`Runtime → Change runtime type → T4 GPU` before running.

## 0. GPU check

Verify a CUDA GPU is available. If not, stop here and switch the runtime.

In [ ]:
import torch
assert torch.cuda.is_available(), (
    "No CUDA GPU detected. Go to Runtime → Change runtime type → T4 GPU, "
    "then Runtime → Restart session and run all."
)
print(f"CUDA OK — {torch.cuda.get_device_name(0)}")
print(f"torch {torch.__version__}, CUDA {torch.version.cuda}")

## 1. Clone the repository

Replace the URL with your GitHub repo. If it's private, use an access token
or make sure the Colab session has access.

In [ ]:
# >>> EDIT THIS to your repo URL <<<
REPO_URL = "https://github.com/DanielSarmiento04/mcrah"
BRANCH = "main"  # change if you train on a different branch

import os, subprocess, sys
if not os.path.exists("/content/mcrah"):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, "/content/mcrah"], check=True)
else:
    subprocess.run(["git", "-C", "/content/mcrah", "pull", "origin", BRANCH], check=True)
os.chdir("/content/mcrah")
if "/content/mcrah/src" not in sys.path:
    sys.path.insert(0, "/content/mcrah/src")
print("working dir:", os.getcwd())
subprocess.run(["git", "log", "-1", "--oneline"])

## 2. Install Python dependencies

Install the package in editable mode so `mcrah` imports work. Colab already
ships torch with CUDA, but we reinstall to be safe and grab the pinned deps
from `requirements.txt`.

In [ ]:
# Colab's bundled torch is CUDA-enabled. Install the project deps on top.
!pip install -e /content/mcrah -q
# lpips is already in pyproject deps, but ensure it's present
!pip install lpips -q

import sys, os, importlib
if "/content/mcrah/src" not in sys.path:
    sys.path.insert(0, "/content/mcrah/src")

# Clear out any stale cached/shadowed mcrah module from memory
for mod_name in list(sys.modules.keys()):
    if mod_name == "mcrah" or mod_name.startswith("mcrah."):
        del sys.modules[mod_name]
importlib.invalidate_caches()

import mcrah
from mcrah.config import Config
print("mcrah import OK — version:", mcrah.__version__, "from", mcrah.__file__)

In [ ]:
# 1. Asegurar dependencias del sistema y cabeceras C++
!apt-get update -qq && apt-get install -y -qq libglm-dev

# 2. Asegurar que los submódulos de git estén descargados completamente
# (Ajusta la ruta si tu carpeta local tiene otro nombre)
!git submodule update --init --recursive

# 3. Configurar variables de entorno para CUDA y arquitectura T4 (sm_75)
import os
import torch

os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["PATH"] = f"/usr/local/cuda/bin:{os.environ['PATH']}"
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"

print(f"PyTorch version: {torch.__version__}")
print(f"PyTorch CUDA version: {torch.version.cuda}")

In [ ]:
# 1. Instalar cabeceras C++ necesarias para la rasterización
!apt-get update -qq && apt-get install -y -qq libglm-dev

# 2. Configurar variables de entorno de CUDA y arquitectura de la GPU T4
import os
os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["PATH"] = f"/usr/local/cuda/bin:{os.environ['PATH']}"
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"

# 3. Clonar con submódulos recursivos y compilar sin aislamiento
# (Se recomienda la versión adaptada de ashawkey o la oficial de inria)
!git clone --recursive https://github.com/ashawkey/diff-gaussian-rasterization.git
!pip install --no-build-isolation -v ./diff-gaussian-rasterization

## 3. Install Rust + build the data preprocessor

The Rust preprocessor parses `transforms_*.json` + PNGs and exports
hypergraph clusters as `.npy` tensors. It's optional (the Python
`DNeRFDataset` reads the raw data directly) but much faster for multi-run
training.

Colab installs Rust via rustup; this takes ~1 min.

In [ ]:
import os, subprocess, pathlib

cargo = pathlib.Path(".cargo/bin/cargo")
if not cargo.exists() and not pathlib.Path("/root/.cargo/bin/cargo").exists():
    print("installing rust toolchain...")
    subprocess.run(
        ["bash", "-c",
         "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y"],
        check=True,
    )

# Make cargo available in this shell session.
os.environ["PATH"] = f"{os.environ['HOME']}/.cargo/bin:{os.environ['PATH']}"
!cargo --version
!rustc --version

In [ ]:
# Build the Rust preprocessor (release). ~30-60 s on Colab.
!cargo build --release --manifest-path rust/Cargo.toml --bin preprocess

## 4. Download the D-NeRF dataset

D-NeRF (Pumarola et al., CVPR 2021) — 8 synthetic dynamic categories.
Official source: https://github.com/albertpumarola/D-NeRF

The full archive is ~3.2 GB and unpacks to `data/` with one folder per
category. We download from the official Dropbox mirror, then verify the
expected structure.

If you only need one category for a smoke test, you can skip the full
download and just pull the files the Python dataset reader needs (see the
cell after next for a single-category shortcut).

In [ ]:
import os, subprocess, pathlib, shutil

# D-NeRF dataset download.
# The dataset is ~260 MB (8 scenes at 502x502 PNG), NOT 3.2 GB.
# Primary: old-format /s/raw/ link (used by nerfstudio's ns-download-data).
# Fallbacks: /scl/fi/ new-format links + Google Drive.
DROPBOX_RAW_URL = "https://www.dropbox.com/s/raw/0bf6fl0ye2vz3vr/data.zip"
DROPBOX_DIRECT_URL = (
    "https://dl.dropboxusercontent.com/scl/fi/cdcmkufncwcikk1dzbgb4/data.zip"
    "?rlkey=n5m21i84v2b2xk6h7qgiu8nkg&dl=1"
)
DROPBOX_FALLBACK_URL = (
    "https://www.dropbox.com/scl/fi/cdcmkufncwcikk1dzbgb4/data.zip"
    "?rlkey=n5m21i84v2b2xk6h7qgiu8nkg&st=y91n1ubj&dl=1"
)
GDRIVE_FILE_ID = "19Na95wk0uikquivC7uKWVqllmTx-mBHt"
ZIP_PATH = "/content/dnerf_data.zip"
DATA_ROOT = pathlib.Path("data")

def _valid_zip(path):
    p = pathlib.Path(path)
    if not p.exists() or p.stat().st_size < 1e8:  # < 100 MB = definitely wrong
        return False
    # Check for ZIP magic bytes (PK\x03\x04) — the definitive test.
    # Dropbox sometimes serves an HTML interstitial instead.
    with open(p, "rb") as f:
        head = f.read(512)
    if head[:4] == b"PK\x03\x04":
        return True  # valid zip header
    if head.lstrip()[:5] in (b"<html", b"<!DOC", b"<!doc", b"<head"):
        return False  # HTML interstitial
    return False

def _try_curl(url, label):
    """Download via curl, print diagnostics, return True on success."""
    print(f"  [{label}] curl -L -o {ZIP_PATH} <url>")
    if pathlib.Path(ZIP_PATH).exists():
        pathlib.Path(ZIP_PATH).unlink()
    r = subprocess.run(["curl", "-L", "--retry", "3", "-o", ZIP_PATH, url],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f"  [{label}] curl exit {r.returncode}")
        print(f"  [{label}] stderr: {r.stderr[-500:]}")
        return False
    sz = pathlib.Path(ZIP_PATH).stat().st_size if pathlib.Path(ZIP_PATH).exists() else 0
    print(f"  [{label}] downloaded {sz / 1e9:.2f} GB")
    if not _valid_zip(ZIP_PATH):
        print(f"  [{label}] file too small or HTML interstitial — not a valid zip")
        return False
    return True

def _download_and_unzip():
    got = False

    # Mirror 1: old-format /s/raw/ link (used by nerfstudio, proven reliable).
    print("downloading D-NeRF dataset (~260 MB) from Dropbox (/s/raw/)...")
    got = _try_curl(DROPBOX_RAW_URL, "dropbox-sraw")

    # Mirror 2: new /scl/fi/ via dl.dropboxusercontent.com (no redirect).
    if not got:
        print("trying Dropbox direct (dl.dropboxusercontent.com)...")
        got = _try_curl(DROPBOX_DIRECT_URL, "dropbox-direct")

    # Mirror 3: new /scl/fi/ via www.dropbox.com with dl=1.
    if not got:
        print("trying Dropbox fallback (www.dropbox.com)...")
        got = _try_curl(DROPBOX_FALLBACK_URL, "dropbox-fallback")

    # Mirror 4: Google Drive via gdown. Google rate-limits this popular
    # file, so this may fail with FileURLRetrievalError.
    if not got:
        print("Dropbox failed. Falling back to gdown (Google Drive)...")
        try:
            subprocess.run(["pip", "install", "--upgrade", "-q", "gdown"],
                           check=True)
            import gdown
            url = f"https://drive.google.com/uc?id={GDRIVE_FILE_ID}"
            gdown.download(url, ZIP_PATH, quiet=False)
            got = _valid_zip(ZIP_PATH)
        except Exception as e:
            print(f"gdown also failed: {e}")

    # Last resort: check if the user manually uploaded the zip.
    if not got and _valid_zip(ZIP_PATH):
        print(f"Found manually uploaded {ZIP_PATH} ({pathlib.Path(ZIP_PATH).stat().st_size / 1e9:.2f} GB)")
        got = True

    if not got or not _valid_zip(ZIP_PATH):
        msg = (
            "Could not download D-NeRF from any mirror. "
            "Manual fallback: download from your browser:\n"
            "  https://www.dropbox.com/scl/fi/cdcmkufncwcikk1dzbgb4/data.zip"
            "?rlkey=n5m21i84v2b2xk6h7qgiu8nkg&st=y91n1ubj&dl=1\n"
            "  or https://drive.google.com/uc?id=19Na95wk0uikquivC7uKWVqllmTx-mBHt\n"
            "Then upload dnerf_data.zip to /content/ and re-run this cell."
        )
        raise RuntimeError(msg)

    print("unzipping...")
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    subprocess.run(["unzip", "-q", ZIP_PATH, "-d", "/content"], check=True)
    # The archive unpacks to /content/data/<category>/... — move into repo.
    src = pathlib.Path("/content/data")
    if src.exists() and src.resolve() != DATA_ROOT.resolve():
        for cat in src.iterdir():
            dest = DATA_ROOT / cat.name
            if not dest.exists():
                shutil.move(str(cat), str(dest))
    if os.path.exists(ZIP_PATH):
        os.remove(ZIP_PATH)
    print("done.")

if not (DATA_ROOT / "trex" / "transforms_train.json").exists():
    _download_and_unzip()
else:
    print("dataset already present, skipping download.")

# Verify structure for the category we will train on.
CATEGORY = "trex"  # change to: bouncingballs, hellwarrior, hook, jumpingjacks, lego, mutant, standup, trex
for f in ("transforms_train.json", "transforms_val.json", "transforms_test.json"):
    assert (DATA_ROOT / CATEGORY / f).exists(), f"missing {DATA_ROOT / CATEGORY / f}"
print(f"dataset OK — training on '{CATEGORY}'")
print("categories present:", sorted(p.name for p in DATA_ROOT.iterdir() if p.is_dir()))

### (Optional) Preprocess with Rust for faster data loading

Run the Rust preprocessor to produce cached `.npy` tensors. This is only
needed if you want to use `ProcessedDNeRFDataset`; the default Python reader
(`DNeRFDataset`) reads raw JSON+PNG and needs no preprocessing.

In [ ]:
# Optional — skip unless you specifically want the .npy cache.
RUN_PREPROCESS = True  # set True to run
if RUN_PREPROCESS:
    !chmod +x ./scripts/preprocess.sh
    !./scripts/preprocess.sh {CATEGORY} --images

## 5. Train MCRAH on one category

Runs the full pipeline via `scripts/train.py`:
- Phase 1: static 3DGS init at t=0
- Phase 3: decoupled two-stage training (dense → farfield → joint)
- Phase 4: evaluation + rollout stability

On Colab's T4 (16 GB VRAM) we keep the cloud small and render resolution low
because the pure-torch rasterizer is O(N·H·W) in autograd memory. These are
the same memory overrides the README recommends for laptops; bump them up if
you have a V100/A100.

In [ ]:

# 2.Configuración de entrenamiento para Google Colab (1 categoría o las 8 del benchmark D-NeRF)
RUN_ALL = True                 # True = entrenar los 8 objetos de D-NeRF; False = solo CATEGORY
CATEGORY = "trex"              # Usado cuando RUN_ALL = False
NUM_GAUSSIANS = 20000          # 20k gaussianas compactas para detalles finos
RENDER_W = 400                 # Ancho de renderizado (puedes subir a 600 u 800 si tienes VRAM)
RENDER_H = 400                 # Alto de renderizado
TIME_WINDOW = 6                # Ventana temporal autorregresiva
ITERATIONS = 1500              # Iteraciones por etapa (dense/farfield/joint)
STATIC_ITERS = 1500            # Iteraciones para Phase 1 (3DGS estático en t=0)
MAX_T0 = 0.06                  # Límite temporal estricto para vistas base t=0
MIN_T0_VIEWS = 12              # Mínimo de cámaras para reconstrucción 3D multi-vista
MAX_SAMPLES = 200              # Frames de entrenamiento a cargar
MAX_EVAL = 50                  # Frames de test para evaluar
ROLLOUT_STEPS = 100            # Longitud del rollout autorregresivo

TARGET_FLAG = "--all" if RUN_ALL else f"--category {CATEGORY}"
!python scripts/train.py \
    {TARGET_FLAG} \
    --num-gaussians {NUM_GAUSSIANS} \
    --render-wh {RENDER_W} {RENDER_H} \
    --time-window {TIME_WINDOW} \
    --iterations {ITERATIONS} \
    --static-iters {STATIC_ITERS} \
    --max-t0 {MAX_T0} \
    --min-t0-views {MIN_T0_VIEWS} \
    --max-samples {MAX_SAMPLES} \
    --max-eval {MAX_EVAL} \
    --rollout-steps {ROLLOUT_STEPS}

## 6. Evaluate + Compare Per-Object Metrics vs. CVPR Baseline

Loads all trained categories in `runs/<category>/results.json`, prints the
per-object comparison table against the official CVPR D-NeRF paper baseline,
generates a ready-to-use LaTeX table, and plots the 100-step rollout drift curves.

In [ ]:
import json, pathlib
from matplotlib import pyplot as plt

CVPR_BASELINES = {
    "bouncingballs": {"psnr": 38.93, "ssim": 0.9870, "lpips": 0.0310},
    "hellwarrior":   {"psnr": 25.02, "ssim": 0.9550, "lpips": 0.0630},
    "hook":          {"psnr": 29.25, "ssim": 0.9680, "lpips": 0.1120},
    "jumpingjacks":  {"psnr": 32.80, "ssim": 0.9810, "lpips": 0.0380},
    "lego":          {"psnr": 21.64, "ssim": 0.8390, "lpips": 0.1630},
    "mutant":        {"psnr": 31.29, "ssim": 0.9780, "lpips": 0.0210},
    "standup":       {"psnr": 32.79, "ssim": 0.9830, "lpips": 0.0240},
    "trex":          {"psnr": 31.75, "ssim": 0.9740, "lpips": 0.0410},
}

all_cats = list(CVPR_BASELINES.keys())
loaded = {}
for c in all_cats:
    rp = pathlib.Path("runs") / c / "results.json"
    if rp.exists():
        with open(rp) as f:
            loaded[c] = json.load(f)

assert loaded, "No results.json files found in runs/ — did training finish?"

print("=" * 98)
print(f"{'OBJECT':15s} | {'PSNR (MCRAH / CVPR)':21s} | {'SSIM (MCRAH / CVPR)':21s} | {'LPIPS (MCRAH / CVPR)':22s} | {'DRIFT@100':9s}")
print("-" * 98)
p_o_list, s_o_list, l_o_list, d_o_list = [], [], [], []
p_b_list, s_b_list, l_b_list = [], [], []

for c, r in loaded.items():
    ev = r.get("eval") or {}
    b = CVPR_BASELINES.get(c, {})
    po, so, lo = ev.get("psnr", 0.0), ev.get("ssim", 0.0), ev.get("lpips", 0.0)
    pb, sb, lb = b.get("psnr", 0.0), b.get("ssim", 0.0), b.get("lpips", 0.0)
    dr = r.get("rollout_final_drift", 0.0)
    p_o_list.append(po); s_o_list.append(so); l_o_list.append(lo); d_o_list.append(dr)
    p_b_list.append(pb); s_b_list.append(sb); l_b_list.append(lb)
    print(f"{c:15s} | {po:6.2f} / {pb:6.2f} dB    | {so:6.4f} / {sb:6.4f}       | {lo:6.4f} / {lb:6.4f}        | {dr:9.5f}")

print("-" * 98)
print(f"{'MEAN':15s} | {sum(p_o_list)/len(p_o_list):6.2f} / {sum(p_b_list)/len(p_b_list):6.2f} dB    | "
      f"{sum(s_o_list)/len(s_o_list):6.4f} / {sum(s_b_list)/len(s_b_list):6.4f}       | "
      f"{sum(l_o_list)/len(l_o_list):6.4f} / {sum(l_b_list)/len(l_b_list):6.4f}        | "
      f"{sum(d_o_list)/len(d_o_list):9.5f}")
print("=" * 98)

# Plot rollout drift curves for all available categories
n_plots = len(loaded)
cols = min(4, n_plots)
rows = (n_plots + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4.5 * cols, 3.0 * rows), squeeze=False)
for idx, (c, r) in enumerate(loaded.items()):
    ax = axes[idx // cols][idx % cols]
    drift = r.get("rollout_pos_drift", [])
    ax.plot(drift, marker=".", markersize=3)
    ax.set_title(f"{c} (drift={r.get('rollout_final_drift', 0):.4f})")
    ax.set_xlabel("step")
    ax.set_ylabel("pos drift")
    ax.grid(True, alpha=0.3)
for idx in range(n_plots, rows * cols):
    axes[idx // cols][idx % cols].axis("off")
plt.tight_layout()
plt.show()

## 7. Render novel views from all trained models vs. Ground Truth

Loads the `checkpoint_joint.pt` for each trained category and renders a
test view side-by-side with Ground Truth.

In [ ]:
import os, sys
if "/content/mcrah/src" not in sys.path:
    sys.path.insert(0, "/content/mcrah/src")
if os.path.exists("/content/mcrah") and os.getcwd() != "/content/mcrah":
    os.chdir("/content/mcrah")

import torch, numpy as np
from pathlib import Path
from matplotlib import pyplot as plt

from mcrah.config import Config
from mcrah.data import DNeRFDataset
from mcrah.training import load_cloud
from mcrah.models import MCRAH
from mcrah.gs import render, set_rasterizer

render_w = RENDER_W if "RENDER_W" in locals() else 400
render_h = RENDER_H if "RENDER_H" in locals() else 400
set_rasterizer("auto")

all_cats = ["bouncingballs", "hellwarrior", "hook", "jumpingjacks", "lego", "mutant", "standup", "trex"]
trained_cats = [c for c in all_cats if (Path("runs") / c / "runs" / "checkpoint_joint.pt").exists()]
assert trained_cats, "No trained checkpoint_joint.pt found in runs/"

for cat in trained_cats:
    cfg = Config.for_category(cat)
    cfg.data.render_wh = (render_w, render_h)
    device = cfg.device_str()
    ckpt_path = Path("runs") / cat / "runs" / "checkpoint_joint.pt"
    cloud_path = Path("runs") / cat / "static_cloud.pt"
    cloud = load_cloud(str(cloud_path), device=device)
    model = MCRAH(cfg, cloud).to(device).eval()
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model"])

    ds = DNeRFDataset(cfg, split="test", categories=[cat])
    sample = ds[0]
    t = torch.tensor([sample.time], device=device)
    bg = torch.ones(3, device=device) if cfg.data.white_background else None
    K = sample.intrinsics.clone().to(device)
    orig_w = float(K[0, 2].item() * 2.0 + 1.0)
    orig_h = float(K[1, 2].item() * 2.0 + 1.0)
    if abs(orig_w - render_w) > 2.0 and orig_w > 0 and orig_h > 0:
        K[0, 0] *= render_w / orig_w
        K[1, 1] *= render_h / orig_h
        K[0, 2] = 0.5 * (render_w - 1.0)
        K[1, 2] = 0.5 * (render_h - 1.0)

    with torch.no_grad():
        steps = model.rollout([t])
        pred_cloud = steps[-1].cloud
        out = render(pred_cloud, c2w=sample.c2w.to(device), intrinsics=K, width=render_w, height=render_h, bg_color=bg)
        pred_img = out.image.clamp(0, 1).cpu().numpy().transpose(1, 2, 0)

    gt = sample.image.numpy().transpose(1, 2, 0)
    fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
    ax[0].imshow(pred_img); ax[0].set_title(f"{cat} — MCRAH render ({render_w}x{render_h})"); ax[0].axis("off")
    ax[1].imshow(gt);       ax[1].set_title(f"{cat} — ground truth"); ax[1].axis("off")
    plt.tight_layout(); plt.show()


## 8. Run the test suite

Optional sanity check that the repo's unit tests pass in this environment.

In [ ]:
!pip install pytest -q
!python -m pytest tests/ -q

## Notes

- **Memory.** The pure-torch rasterizer is O(N·H·W) in autograd memory. If
  you hit OOM, lower `--num-gaussians`, `--render-wh`, or `--time-window`.
  On T4 (16 GB) the defaults above (~20k Gaussians, 400×400, T=8) are
  near the safe ceiling; on A100/V100 you can go to 50k Gaussians at
  800×800.
- **Rust is optional.** The Python `DNeRFDataset` reads raw JSON+PNG, so
  you can skip sections 3 entirely for a smoke test. The Rust preprocessor
  only matters if you want cached `.npy` tensors for fast re-runs.
- **Dataset source.** D-NeRF is from [Pumarola et al., CVPR 2021](https://github.com/albertpumarola/D-NeRF); download links are in that repo's README.